# Core CRUD Workflow


**Workflows covered:**
1. Setup & Engine Connection
2. Read `friends` & `books` tables (+ test after each)
3. Create a friend & loan using dynamically selected row data (+ test)
4. Update a friend using dynamic selection (+ test)
5. Delete a friend using dynamic selection (+ test)


## 0. Setup & Engine Connection

In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

# Running this cell REQUIRES a module named con_lib.py in this notebook's PARENT DIRECTORY
# ALTERNATELY, provide a connection string in a different way
#schema = "Your_Schema"
#host = "127.0.0.1"
##user = "root"
#password = "YOUR_PASSWORD"
##port = 3306

#connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

import sys
sys.path.append("..") # con_lib is the parent directory of this notebook
from Con_lib1 import connection_string

engine = create_engine(connection_string)


Read from sql

In [4]:
pd.read_sql('books', con=engine)

,isbn,title,author,genre,book_language,is_available
0,9780061120084,To Kill a Mockingbird,Harper Lee,Classic,English,1
1,9780132350884,Clean Code,Robert C. Martin,Programming,English,1
2,9780140449136,The Odyssey,Homer,Epic,English,1
3,9780261103303,The Hobbit,J.R.R. Tolkien,Fantasy,English,1
4,9780439023481,The Hunger Games,Suzanne Collins,Science Fiction,English,1
5,9780451524935,1984,George Orwell,Dystopian,English,1
6,9780553386790,Dune,Frank Herbert,Science Fiction,English,1
7,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Classic,English,1
8,9781491957660,Python Crash Course,Eric Matthes,Programming,English,1
9,9784062745000,Norwegian Wood,Haruki Murakami,Romance,Japanese,1


In [5]:
pd.read_sql('users', con=connection_string)

,user_id,user_name,phone_number,email,max_loans
0,1,John Smith,9876543210,john.smith@gmail.com,5
1,2,John shaw,49157559998,johnshaw@email.com,3
2,3,Michael Brown,9876543212,michael.brown@gmail.com,3
3,4,Sophia Davis,9876543213,sophia.davis@gmail.com,5
4,5,Daniel Wilson,9876543214,daniel.wilson@gmail.com,4
5,6,Olivia Martinez,9876543215,olivia.martinez@gmail.com,5
6,7,James Anderson,9876543216,james.anderson@gmail.com,2
7,8,Ava Taylor,9876543217,ava.taylor@gmail.com,5
8,9,William Thomas,9876543218,william.thomas@gmail.com,3
9,10,Isabella Moore,9876543219,isabella.moore@gmail.com,5


In [6]:
pd.read_sql('loans', con=connection_string)


,loan_id,isbn,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date
0,1,9780140449136,1,2026-09-01,2026-09-15,NaT,0,0.0,NaT
1,2,9780743273565,2,2026-09-02,2026-09-16,NaT,0,0.0,NaT
2,3,9780061120084,3,2026-08-20,2026-09-03,2026-09-10,1,2.5,NaT
3,4,9780451524935,4,2026-09-04,2026-09-18,NaT,0,0.0,NaT
4,5,9780261103303,5,2026-08-25,2026-09-08,NaT,0,5.0,NaT
5,6,9780439023481,6,2026-09-05,2026-09-19,NaT,0,0.0,NaT
6,7,9780132350884,7,2026-08-15,2026-08-29,NaT,0,10.0,2026-08-28
7,8,9781491957660,8,2026-09-06,2026-09-20,NaT,0,0.0,NaT
8,9,9784062745000,9,2026-08-10,2026-08-24,NaT,0,7.5,2026-08-23
9,10,9788184001754,10,2026-09-07,2026-09-21,NaT,0,0.0,NaT


In [7]:
pd.read_sql('reservations', con=connection_string)

,reservation_id,isbn,user_id,reservation_date,ready_date,status
0,1,9780140449136,2,2026-09-08 10:00:00,NaT,WAIT
1,2,9780743273565,3,2026-09-08 11:30:00,NaT,WAIT
2,3,9780061120084,4,2026-09-08 12:00:00,2026-09-10 09:00:00,READY
3,4,9780451524935,5,2026-09-09 09:15:00,NaT,WAIT
4,5,9780261103303,6,2026-09-09 10:30:00,2026-09-11 14:00:00,READY
5,6,9780439023481,7,2026-09-09 13:00:00,NaT,WAIT
6,7,9780132350884,8,2026-09-10 09:45:00,NaT,CANCELLED
7,8,9781491957660,9,2026-09-10 11:00:00,2026-09-12 10:00:00,FULFILLED
8,9,9784062745000,10,2026-09-10 15:30:00,NaT,EXPIRED
9,10,9788184001754,1,2026-09-11 16:00:00,NaT,WAIT


## 1. Read Operations (Users, loans,books, reservations)

In [8]:
def read_users():
    return pd.read_sql('users', con=engine)

In [9]:
read_users()

,user_id,user_name,phone_number,email,max_loans
0,1,John Smith,9876543210,john.smith@gmail.com,5
1,2,John shaw,49157559998,johnshaw@email.com,3
2,3,Michael Brown,9876543212,michael.brown@gmail.com,3
3,4,Sophia Davis,9876543213,sophia.davis@gmail.com,5
4,5,Daniel Wilson,9876543214,daniel.wilson@gmail.com,4
5,6,Olivia Martinez,9876543215,olivia.martinez@gmail.com,5
6,7,James Anderson,9876543216,james.anderson@gmail.com,2
7,8,Ava Taylor,9876543217,ava.taylor@gmail.com,5
8,9,William Thomas,9876543218,william.thomas@gmail.com,3
9,10,Isabella Moore,9876543219,isabella.moore@gmail.com,5


In [10]:
def read_books(available_only=False):
    books = pd.read_sql("books", con=engine)
    loans= pd.read_sql("loans", con=engine)

    if available_only == True:
        books = pd.merge(books, loans, on='isbn', how='left').query('user_id.isna()')[books.columns]

    return books

In [11]:
read_books(available_only=True)

,isbn,title,author,genre,book_language,is_available


In [12]:
read_books()

,isbn,title,author,genre,book_language,is_available
0,9780061120084,To Kill a Mockingbird,Harper Lee,Classic,English,1
1,9780132350884,Clean Code,Robert C. Martin,Programming,English,1
2,9780140449136,The Odyssey,Homer,Epic,English,1
3,9780261103303,The Hobbit,J.R.R. Tolkien,Fantasy,English,1
4,9780439023481,The Hunger Games,Suzanne Collins,Science Fiction,English,1
5,9780451524935,1984,George Orwell,Dystopian,English,1
6,9780553386790,Dune,Frank Herbert,Science Fiction,English,1
7,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Classic,English,1
8,9781491957660,Python Crash Course,Eric Matthes,Programming,English,1
9,9784062745000,Norwegian Wood,Haruki Murakami,Romance,Japanese,1


In [13]:
books = pd.read_sql("books", con=engine)
loans= pd.read_sql("loans", con=engine)

available_books = pd.merge(books, loans, on='isbn', how='left').query('user_id.isna()')

available_books

,isbn,title,author,genre,book_language,is_available,loan_id,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date


In [14]:
available_books

,isbn,title,author,genre,book_language,is_available,loan_id,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date


In [15]:
read_books()

,isbn,title,author,genre,book_language,is_available
0,9780061120084,To Kill a Mockingbird,Harper Lee,Classic,English,1
1,9780132350884,Clean Code,Robert C. Martin,Programming,English,1
2,9780140449136,The Odyssey,Homer,Epic,English,1
3,9780261103303,The Hobbit,J.R.R. Tolkien,Fantasy,English,1
4,9780439023481,The Hunger Games,Suzanne Collins,Science Fiction,English,1
5,9780451524935,1984,George Orwell,Dystopian,English,1
6,9780553386790,Dune,Frank Herbert,Science Fiction,English,1
7,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Classic,English,1
8,9781491957660,Python Crash Course,Eric Matthes,Programming,English,1
9,9784062745000,Norwegian Wood,Haruki Murakami,Romance,Japanese,1


In [16]:
def read_loans():
    return pd.read_sql("loans", con=engine)

In [17]:
read_loans()

,loan_id,isbn,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date
0,1,9780140449136,1,2026-09-01,2026-09-15,NaT,0,0.0,NaT
1,2,9780743273565,2,2026-09-02,2026-09-16,NaT,0,0.0,NaT
2,3,9780061120084,3,2026-08-20,2026-09-03,2026-09-10,1,2.5,NaT
3,4,9780451524935,4,2026-09-04,2026-09-18,NaT,0,0.0,NaT
4,5,9780261103303,5,2026-08-25,2026-09-08,NaT,0,5.0,NaT
5,6,9780439023481,6,2026-09-05,2026-09-19,NaT,0,0.0,NaT
6,7,9780132350884,7,2026-08-15,2026-08-29,NaT,0,10.0,2026-08-28
7,8,9781491957660,8,2026-09-06,2026-09-20,NaT,0,0.0,NaT
8,9,9784062745000,9,2026-08-10,2026-08-24,NaT,0,7.5,2026-08-23
9,10,9788184001754,10,2026-09-07,2026-09-21,NaT,0,0.0,NaT


## 1. Read users

In [18]:

def read_users(engine):
    users = pd.read_sql(
        """
        SELECT
            user_id,
            user_name,
            phone_number,
            email,
            max_loans
        FROM users
        ORDER BY user_id
        """,
        con=engine
    )

    return users

In [19]:
read_users(engine)

,user_id,user_name,phone_number,email,max_loans
0,1,John Smith,9876543210,john.smith@gmail.com,5
1,2,John shaw,49157559998,johnshaw@email.com,3
2,3,Michael Brown,9876543212,michael.brown@gmail.com,3
3,4,Sophia Davis,9876543213,sophia.davis@gmail.com,5
4,5,Daniel Wilson,9876543214,daniel.wilson@gmail.com,4
5,6,Olivia Martinez,9876543215,olivia.martinez@gmail.com,5
6,7,James Anderson,9876543216,james.anderson@gmail.com,2
7,8,Ava Taylor,9876543217,ava.taylor@gmail.com,5
8,9,William Thomas,9876543218,william.thomas@gmail.com,3
9,10,Isabella Moore,9876543219,isabella.moore@gmail.com,5


## 2. Create users


In [20]:
read_users()

TypeError: read_users() missing 1 required positional argument: 'engine'

In [ ]:
def create_users(user_name, max_loans=5, phone_number=None, email=None):
    df = pd.DataFrame(
        [[user_name, max_loans, phone_number, email]],
        columns=["user_name", "max_loans", "phone_number", "email"]
    )

    try:
        df.to_sql(
            "users",
            if_exists="append",
            con=engine,
            index=False
        )
        return f"Added {user_name} to users table"

    except Exception as error:
        print("Database error:", error)
        raise

In [ ]:
create_users("John Black",3,"49157557967","johnblack@email.com")

'Added John Black to users table'

In [ ]:
read_users()

,user_id,user_name,phone_number,email,max_loans
0,1,John Smith,9876543210,john.smith@gmail.com,5
1,2,John shaw,49157559998,johnshaw@email.com,3
2,3,Michael Brown,9876543212,michael.brown@gmail.com,3
3,4,Sophia Davis,9876543213,sophia.davis@gmail.com,5
4,5,Daniel Wilson,9876543214,daniel.wilson@gmail.com,4
5,6,Olivia Martinez,9876543215,olivia.martinez@gmail.com,5
6,7,James Anderson,9876543216,james.anderson@gmail.com,2
7,8,Ava Taylor,9876543217,ava.taylor@gmail.com,5
8,9,William Thomas,9876543218,william.thomas@gmail.com,3
9,10,Isabella Moore,9876543219,isabella.moore@gmail.com,5


## 3. Update Users

In [ ]:
from sqlalchemy import text

def update_user(
    user_id,
    user_name=None,
    phone_number=None,
    email=None,
    max_loans=None
):
    updates = {}
    set_clauses = []

    if user_name is not None:
        set_clauses.append("user_name = :user_name")
        updates["user_name"] = user_name

    if phone_number is not None:
        set_clauses.append("phone_number = :phone_number")
        updates["phone_number"] = phone_number

    if email is not None:
        set_clauses.append("email = :email")
        updates["email"] = email

    if max_loans is not None:
        set_clauses.append("max_loans = :max_loans")
        updates["max_loans"] = max_loans

    if not set_clauses:
        return "No changes provided"

    updates["user_id"] = user_id

    query = text(f"""
        UPDATE users
        SET {", ".join(set_clauses)}
        WHERE user_id = :user_id
    """)

    try:
        with engine.begin() as connection:
            result = connection.execute(query, updates)

        if result.rowcount == 0:
            return f"No user found with user_id {user_id}"

        return f"User {user_id} updated successfully"

    except Exception as error:
        print("Database error:", error)
        raise

In [ ]:
update_user(
    user_id=2,
    user_name="John shaw",
    phone_number="49157559998",
    email="johnshaw@email.com",
    max_loans=3
)

'User 2 updated successfully'

In [ ]:
def read_users():
    query = "SELECT * FROM users"
    return pd.read_sql(query, con=engine)

In [ ]:
read_users()

,user_id,user_name,phone_number,email,max_loans
0,1,John Smith,9876543210,john.smith@gmail.com,5
1,2,John shaw,49157559998,johnshaw@email.com,3
2,3,Michael Brown,9876543212,michael.brown@gmail.com,3
3,4,Sophia Davis,9876543213,sophia.davis@gmail.com,5
4,5,Daniel Wilson,9876543214,daniel.wilson@gmail.com,4
5,6,Olivia Martinez,9876543215,olivia.martinez@gmail.com,5
6,7,James Anderson,9876543216,james.anderson@gmail.com,2
7,8,Ava Taylor,9876543217,ava.taylor@gmail.com,5
8,9,William Thomas,9876543218,william.thomas@gmail.com,3
9,10,Isabella Moore,9876543219,isabella.moore@gmail.com,5


## 4. Delete User

In [ ]:
from sqlalchemy import text


def delete_user(user_id, engine):

    with engine.begin() as connection:

        # Check whether the user exists
        user_exists = connection.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM users
                    WHERE user_id = :user_id
                )
            """),
            {"user_id": user_id}
        ).scalar()

        if not user_exists:
            return f"User {user_id} does not exist."

        # Check whether the user has any loans
        loan_check = connection.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM loans
                    WHERE user_id = :user_id
                )
            """),
            {"user_id": user_id}
        ).scalar()

        # Check whether the user has any reservations
        reservation_check = connection.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM reservations
                    WHERE user_id = :user_id
                )
            """),
            {"user_id": user_id}
        ).scalar()

        # Prevent deletion if either record exists
        if loan_check and reservation_check:
            return (
                "User cannot be deleted because they have "
                "a loan and a reservation."
            )

        elif loan_check:
            return "User cannot be deleted because they have a loan."

        elif reservation_check:
            return "User cannot be deleted because they have a reservation."

        # Delete the user
        result = connection.execute(
            text("""
                DELETE FROM users
                WHERE user_id = :user_id
            """),
            {"user_id": user_id}
        )

        # Check whether a row was actually deleted
        if result.rowcount == 1:
            return f"User {user_id} deleted successfully."

        return f"User {user_id} was not deleted."

In [ ]:
read_users()

,user_id,user_name,phone_number,email,max_loans
0,1,John Smith,9876543210,john.smith@gmail.com,5
1,2,John shaw,49157559998,johnshaw@email.com,3
2,3,Michael Brown,9876543212,michael.brown@gmail.com,3
3,4,Sophia Davis,9876543213,sophia.davis@gmail.com,5
4,5,Daniel Wilson,9876543214,daniel.wilson@gmail.com,4
5,6,Olivia Martinez,9876543215,olivia.martinez@gmail.com,5
6,7,James Anderson,9876543216,james.anderson@gmail.com,2
7,8,Ava Taylor,9876543217,ava.taylor@gmail.com,5
8,9,William Thomas,9876543218,william.thomas@gmail.com,3
9,10,Isabella Moore,9876543219,isabella.moore@gmail.com,5


In [ ]:
delete_user(99, engine)

'User 99 does not exist.'

In [ ]:
delete_user(26, engine)

'User 26 deleted successfully.'

In [ ]:
delete_user(2, engine)

'User cannot be deleted because they have a loan and a reservation.'

Update , create , deleted and modify loans


In [ ]:
import pandas as pd


def read_loans(engine):
    loans = pd.read_sql(
        """
        SELECT
            loan_id,
            isbn,
            user_id,
            loan_date,
            due_date,
            extended_due_date,
            extension_count,
            late_fee,
            return_date
        FROM loans
        ORDER BY loan_id
        """,
        con=engine
    )

    return loans

In [ ]:
read_loans(engine)

,loan_id,isbn,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date
0,1,9780140449136,1,2026-09-01,2026-09-15,None,0,0.0,None
1,2,9780743273565,2,2026-09-02,2026-09-16,None,0,0.0,None
2,3,9780061120084,3,2026-08-20,2026-09-03,2026-09-10,1,2.5,None
3,4,9780451524935,4,2026-09-04,2026-09-18,None,0,0.0,None
4,5,9780261103303,5,2026-08-25,2026-09-08,None,0,5.0,None
5,6,9780439023481,6,2026-09-05,2026-09-19,None,0,0.0,None
6,7,9780132350884,7,2026-08-15,2026-08-29,None,0,10.0,2026-08-28
7,8,9781491957660,8,2026-09-06,2026-09-20,None,0,0.0,None
8,9,9784062745000,9,2026-08-10,2026-08-24,None,0,7.5,2026-08-23
9,10,9788184001754,10,2026-09-07,2026-09-21,None,0,0.0,None


In [ ]:
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import text


def create_loan(
    isbn,
    user_id,
    loan_date=None,
    engine=None,
    extended_due_date=None,
    extension_count=0,
    late_fee=0.00,
    return_date=None
):
    # If no loan date is provided, use today's date
    if loan_date is None:
        loan_date = date.today()

    # Due date is automatically 3 weeks after the loan date
    due_date = loan_date + timedelta(weeks=3)

    with engine.begin() as connection:

        # Check whether the book exists
        book_exists = connection.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM books
                    WHERE isbn = :isbn
                )
            """),
            {"isbn": isbn}
        ).scalar()

        if not book_exists:
            return f"Book with ISBN {isbn} does not exist."

        # Check whether the user exists
        user_exists = connection.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM users
                    WHERE user_id = :user_id
                )
            """),
            {"user_id": user_id}
        ).scalar()

        if not user_exists:
            return f"User {user_id} does not exist."

        # Check whether the book is already loaned
        active_loan = connection.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM loans
                    WHERE isbn = :isbn
                    AND return_date IS NULL
                )
            """),
            {"isbn": isbn}
        ).scalar()

        if active_loan:
            return (
                f"Book with ISBN {isbn} is already loaned "
                "and cannot be loaned again."
            )

        # Create the loan
        df = pd.DataFrame(
            [[
                isbn,
                user_id,
                loan_date,
                due_date,
                extended_due_date,
                extension_count,
                late_fee,
                return_date
            ]],
            columns=[
                "isbn",
                "user_id",
                "loan_date",
                "due_date",
                "extended_due_date",
                "extension_count",
                "late_fee",
                "return_date"
            ]
        )

        df.to_sql(
            "loans",
            con=connection,
            if_exists="append",
            index=False
        )

        return (
            f"Loan created for user {user_id}, book ISBN {isbn}. "
            f"Loan date: {loan_date}; Due date: {due_date} "
            f"(3 weeks after loan date)."
        )

In [ ]:
result = create_loan(
    isbn="9780132350884",
    user_id=1,
    loan_date=date.today(),
    engine=engine
)

print(result)

Loan created for user 1, book ISBN 9780132350884. Loan date: 2026-09-15; Due date: 2026-10-06 (3 weeks after loan date).


In [ ]:
read_loans(engine)

,loan_id,isbn,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date
0,1,9780140449136,1,2026-09-01,2026-09-15,None,0,0.0,None
1,2,9780743273565,2,2026-09-02,2026-09-16,None,0,0.0,None
2,3,9780061120084,3,2026-08-20,2026-09-03,2026-09-10,1,2.5,None
3,4,9780451524935,4,2026-09-04,2026-09-18,None,0,0.0,None
4,5,9780261103303,5,2026-08-25,2026-09-08,None,0,5.0,None
5,6,9780439023481,6,2026-09-05,2026-09-19,None,0,0.0,None
6,7,9780132350884,7,2026-08-15,2026-08-29,None,0,10.0,2026-08-28
7,8,9781491957660,8,2026-09-06,2026-09-20,None,0,0.0,None
8,9,9784062745000,9,2026-08-10,2026-08-24,None,0,7.5,2026-08-23
9,10,9788184001754,10,2026-09-07,2026-09-21,None,0,0.0,None


Delete From loans

In [21]:
from sqlalchemy import text


def delete_loan(loan_id, engine):
    query = text("""
        DELETE FROM loans
        WHERE loan_id = :loan_id
          AND return_date IS NOT NULL
    """)

    try:
        with engine.begin() as connection:
            result = connection.execute(
                query,
                {"loan_id": loan_id}
            )

        if result.rowcount == 0:
            return (
                f"Loan {loan_id} does not exist "
                "or the book has not been returned yet."
            )

        return f"Loan {loan_id} deleted successfully."

    except Exception as error:
        print("Database error:", error)
        raise

In [22]:
delete_loan(1, engine)

'Loan 1 does not exist or the book has not been returned yet.'

In [ ]:
delete_loan(99, engine)

'Loan 99 does not exist.'

In [ ]:
read_loans(engine)

,loan_id,isbn,user_id,loan_date,due_date,extended_due_date,extension_count,late_fee,return_date
0,1,9780140449136,1,2026-09-01,2026-09-15,None,0,0.0,None
1,2,9780743273565,2,2026-09-02,2026-09-16,None,0,0.0,None
2,3,9780061120084,3,2026-08-20,2026-09-03,2026-09-10,1,2.5,None
3,4,9780451524935,4,2026-09-04,2026-09-18,None,0,0.0,None
4,5,9780261103303,5,2026-08-25,2026-09-08,None,0,5.0,None
5,6,9780439023481,6,2026-09-05,2026-09-19,None,0,0.0,None
6,7,9780132350884,7,2026-08-15,2026-08-29,None,0,10.0,2026-08-28
7,8,9781491957660,8,2026-09-06,2026-09-20,None,0,0.0,None
8,9,9784062745000,9,2026-08-10,2026-08-24,None,0,7.5,2026-08-23
9,10,9788184001754,10,2026-09-07,2026-09-21,None,0,0.0,None


modify the loans

In [23]:
import pandas as pd
from datetime import date, datetime
from sqlalchemy import text


def modify_loan(loan_id, engine):

    choice = input(
        "What do you want to update?\n"
        "1. Extend due date\n"
        "2. Calculate/update late fee\n"
        "3. Add return date\n"
        "Enter 1, 2, or 3: "
    ).strip()

    with engine.begin() as connection:

        loan = connection.execute(
            text("""
                SELECT
                    loan_id,
                    due_date,
                    extended_due_date,
                    extension_count,
                    late_fee,
                    return_date
                FROM loans
                WHERE loan_id = :loan_id
            """),
            {"loan_id": loan_id}
        ).mappings().first()

        if loan is None:
            return f"Loan {loan_id} does not exist."

        effective_due_date = (
            loan["extended_due_date"]
            or loan["due_date"]
        )

        # ---------------------------------------------
        # Option 1: Extend due date
        # ---------------------------------------------
        if choice == "1":

            if loan["return_date"] is not None:
                return (
                    f"Loan {loan_id} has already been returned. "
                    "Further extension is not possible."
                )

            if loan["extension_count"] >= 3:
                return (
                    f"Loan {loan_id} has already been extended 3 times. "
                    "Further extension is not possible."
                )

            new_date_input = input(
                "Enter the new extended due date (YYYY-MM-DD): "
            ).strip()

            try:
                new_extended_due_date = datetime.strptime(
                    new_date_input,
                    "%Y-%m-%d"
                ).date()
            except ValueError:
                return "Invalid date format. Use YYYY-MM-DD."

            if new_extended_due_date <= effective_due_date:
                return (
                    "The new extended due date must be later than "
                    f"{effective_due_date}."
                )

            new_extension_count = loan["extension_count"] + 1

            connection.execute(
                text("""
                    UPDATE loans
                    SET
                        extended_due_date = :extended_due_date,
                        extension_count = :extension_count
                    WHERE loan_id = :loan_id
                """),
                {
                    "extended_due_date": new_extended_due_date,
                    "extension_count": new_extension_count,
                    "loan_id": loan_id
                }
            )

            return (
                f"Loan {loan_id} updated successfully.\n"
                f"Extended due date: {new_extended_due_date}\n"
                f"Extension count: {new_extension_count}/3"
            )

        # ---------------------------------------------
        # Option 2: Calculate/update late fee
        # ---------------------------------------------
        elif choice == "2":

            end_date = loan["return_date"] or date.today()

            late_days = max(
                0,
                (end_date - effective_due_date).days
            )

            late_fee = late_days * 0.50

            connection.execute(
                text("""
                    UPDATE loans
                    SET late_fee = :late_fee
                    WHERE loan_id = :loan_id
                """),
                {
                    "late_fee": late_fee,
                    "loan_id": loan_id
                }
            )

            return (
                f"Loan {loan_id} late fee updated successfully.\n"
                f"Late days: {late_days}\n"
                f"Late fee: ${late_fee:.2f}"
            )

        # ---------------------------------------------
        # Option 3: Add return date
        # ---------------------------------------------
        elif choice == "3":

            if loan["return_date"] is not None:
                return (
                    f"Loan {loan_id} already has a return date: "
                    f"{loan['return_date']}."
                )

            return_date_input = input(
                "Enter the return date (YYYY-MM-DD): "
            ).strip()

            try:
                new_return_date = datetime.strptime(
                    return_date_input,
                    "%Y-%m-%d"
                ).date()
            except ValueError:
                return "Invalid date format. Use YYYY-MM-DD."

            late_days = max(
                0,
                (new_return_date - effective_due_date).days
            )

            late_fee = late_days * 0.50

            connection.execute(
                text("""
                    UPDATE loans
                    SET
                        return_date = :return_date,
                        late_fee = :late_fee
                    WHERE loan_id = :loan_id
                """),
                {
                    "return_date": new_return_date,
                    "late_fee": late_fee,
                    "loan_id": loan_id
                }
            )

            return (
                f"Loan {loan_id} returned successfully.\n"
                f"Return date: {new_return_date}\n"
                f"Late days: {late_days}\n"
                f"Late fee: ${late_fee:.2f}"
            )

        else:
            return "Invalid choice. Please enter 1, 2, or 3."

In [24]:
result = modify_loan(loan_id=3, engine=engine)
print(result)

Invalid date format. Use YYYY-MM-DD.


Update , create , delete the books

In [25]:
def create_book(
    isbn,
    title,
    author,
    genre,
    book_language,
    is_available,
    engine
):
    query = text("""
        INSERT INTO books (
            isbn,
            title,
            author,
            genre,
            book_language,
            is_available
        )
        VALUES (
            :isbn,
            :title,
            :author,
            :genre,
            :book_language,
            :is_available
        )
    """)

    try:
        with engine.begin() as connection:

            book_exists = connection.execute(
                text("""
                    SELECT EXISTS (
                        SELECT 1
                        FROM books
                        WHERE isbn = :isbn
                    )
                """),
                {"isbn": isbn}
            ).scalar()

            if book_exists:
                return f"Book with ISBN {isbn} already exists."

            connection.execute(
                query,
                {
                    "isbn": isbn,
                    "title": title,
                    "author": author,
                    "genre": genre,
                    "book_language": book_language,
                    "is_available": is_available
                }
            )

        return f"Book '{title}' created successfully."

    except Exception as error:
        print("Database error:", error)
        raise

In [26]:
print(create_book(
    isbn="9781234567890",
    title="Introduction to Databases",
    author="John Smith",
    genre="Programming",
    book_language="English",
    is_available=1,
    engine=engine
))

Book 'Introduction to Databases' created successfully.


In [27]:
def update_book(isbn, engine):

    print("What do you want to update?")
    print("1. Title")
    print("2. Author")
    print("3. Genre")
    print("4. Book language")
    print("5. Availability")

    choice = input("Enter your choice: ").strip()

    fields = {
        "1": "title",
        "2": "author",
        "3": "genre",
        "4": "book_language",
        "5": "is_available"
    }

    if choice not in fields:
        return "Invalid choice."

    field = fields[choice]

    if field == "is_available":
        new_value = input(
            "Enter availability (1 = available, 0 = unavailable): "
        ).strip()

        if new_value not in ["0", "1"]:
            return "Availability must be 0 or 1."

        new_value = int(new_value)

    else:
        new_value = input(
            f"Enter the new value for {field}: "
        ).strip()

        if new_value == "":
            return "Value cannot be empty."

    query = text(f"""
        UPDATE books
        SET {field} = :new_value
        WHERE isbn = :isbn
    """)

    try:
        with engine.begin() as connection:
            result = connection.execute(
                query,
                {
                    "new_value": new_value,
                    "isbn": isbn
                }
            )

        if result.rowcount == 0:
            return f"Book with ISBN {isbn} does not exist."

        return f"Book with ISBN {isbn} updated successfully."

    except Exception as error:
        print("Database error:", error)
        raise

In [28]:
print(update_book(
    isbn="9781234567890",
    engine=engine
))

What do you want to update?
1. Title
2. Author
3. Genre
4. Book language
5. Availability
Book with ISBN 9781234567890 updated successfully.


Delete Books

In [29]:
def delete_book(isbn, engine):

    try:
        with engine.begin() as connection:

            book_exists = connection.execute(
                text("""
                    SELECT EXISTS (
                        SELECT 1
                        FROM books
                        WHERE isbn = :isbn
                    )
                """),
                {"isbn": isbn}
            ).scalar()

            if not book_exists:
                return f"Book with ISBN {isbn} does not exist."

            loan_exists = connection.execute(
                text("""
                    SELECT EXISTS (
                        SELECT 1
                        FROM loans
                        WHERE isbn = :isbn
                    )
                """),
                {"isbn": isbn}
            ).scalar()

            if loan_exists:
                return (
                    f"Book with ISBN {isbn} cannot be deleted "
                    "because loan records exist for this book."
                )

            result = connection.execute(
                text("""
                    DELETE FROM books
                    WHERE isbn = :isbn
                """),
                {"isbn": isbn}
            )

        if result.rowcount == 0:
            return f"Book with ISBN {isbn} was not deleted."

        return f"Book with ISBN {isbn} deleted successfully."

    except Exception as error:
        print("Database error:", error)
        raise

In [30]:
print(delete_book(
    isbn="9781234567890",
    engine=engine
))

Book with ISBN 9781234567890 deleted successfully.


In [31]:
def read_books(engine):

    books = pd.read_sql(
        """
        SELECT
            isbn,
            title,
            author,
            genre,
            book_language,
            is_available
        FROM books
        ORDER BY isbn
        """,
        con=engine
    )

    return books

In [32]:
books_df = read_books(engine)
books_df

,isbn,title,author,genre,book_language,is_available
0,9780061120084,To Kill a Mockingbird,Harper Lee,Classic,English,1
1,9780132350884,Clean Code,Robert C. Martin,Programming,English,1
2,9780140449136,The Odyssey,Homer,Epic,English,1
3,9780261103303,The Hobbit,J.R.R. Tolkien,Fantasy,English,1
4,9780439023481,The Hunger Games,Suzanne Collins,Science Fiction,English,1
5,9780451524935,1984,George Orwell,Dystopian,English,1
6,9780553386790,Dune,Frank Herbert,Science Fiction,English,1
7,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Classic,English,1
8,9781491957660,Python Crash Course,Eric Matthes,Programming,English,1
9,9784062745000,Norwegian Wood,Haruki Murakami,Romance,Japanese,1
